0. 파일 분석 후 병합
1.파일 읽어오기
2.기본 정보 및 기초 통계량 확인
3.한글 컬럼명 영문명으로 변경
4.데이터 타입 변경
5.결측치 확인 및 처리
6.상관계수 함수를 이용하여 요소별 상관관계 분석
7.히스토그램으로 시각화
8.막대그래프로 일별 현황 그래프 출력
9.히트맵으로 상관관계 시각화
10.산점도 그래프로 온도와 미세먼지(PM10) 확인
11.미세먼지(PM10) 과 초미세먼지(PM2.5) 관계를 산점도 그래프로 확인
12.데이터 분석 정리

In [1]:
# 1. 라이브러리 import
import pandas as pd #데이터를 표 형태로 다루기 위한 라이브러리
pd.set_option('display.max_columns', None) #데이터프레임 출력 시 모든 열을 표시하도록 설정
pd.set_option('display.max_rows', None) #데이터프레임 출력 시 모든 행을 표시하도록 설정
pd.set_option('display.width', None)  #데이터프레임 출력 시 너비 제한을 없앰
pd.set_option('display.max_colwidth', None) #데이터프레임 출력 시 열의 최대 너비 제한을 없앰

import os #운영체제에서 제공하는 기능을 사용하기 위한 라이브러리
import numpy as np #수치 계산을 위한 라이브러리
import matplotlib.pyplot as plt #데이터 시각화를 위한 라이브러리
import seaborn as sns #데이터 시각화를 위한 라이브러리

In [2]:
# 2. 한글 폰트 설치 및 설정 : Jupyter Notebook 환경에서 한글 깨짐 방지를 위한 폰트 설정
!apt-get install -y fonts-nanum
!fc-cache -fv
!rm ~/.cache/matplotlib -rf

plt.rc('font', family='NanumBarunGothic')
plt.rcParams['axes.unicode_minus'] = False #마이너스 기호 깨짐 방지 설정

print("설치 완료")


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
fonts-nanum is already the newest version (20200506-1).
0 upgraded, 0 newly installed, 0 to remove and 0 not upgraded.
/bin/bash: line 1: fc-cache: command not found
설치 완료


In [ ]:
# 3. 파일 기본정보 확인
def detect_encoding(filepath):
    """파일의 인코딩을 탐지"""
    encodings_to_test = ['utf-8', 'cp949', 'euc-kr', 'latin-1']
    
    for enc in encodings_to_test:
        try:
            with open(filepath, 'r', encoding=enc) as f:
                f.read()
            return enc
        except:
            continue
    return "Unknown"

file_list = ['/workspace/Python/final_exam/data/weather.csv', 
             '/workspace/Python/final_exam/data/measuredData.csv']

for filename in file_list:
    if os.path.exists(filename):
        file_size = os.path.getsize(filename)
        file_ext = os.path.splitext(filename)[1]
        file_encoding = detect_encoding(filename)
        print(f"파일명: {filename}")
        print(f"확장자: {file_ext}")
        print(f"파일 크기: {file_size} bytes")
        print(f"인코딩: {file_encoding}")
        print()


파일명: /workspace/Python/final_exam/data/weather.csv
확장자: .csv
파일 크기: 30430 bytes
인코딩: utf-8

파일명: /workspace/Python/final_exam/data/measuredData.csv
확장자: .csv
파일 크기: 31361 bytes
인코딩: utf-8



In [4]:
# # 4. weather.csv 인코딩(cp949) 문제를 해결하기 위해 utf-8로 변환하여 파일 덮어쓰기 (한 번만 실행)
# weather = pd.read_csv('/workspace/Python/final_exam/data/weather.csv', encoding='cp949')
# weather.to_csv('weather.csv', encoding='utf-8', index=False)

# print("저장 완료")

# #3번 셀 재실행해서 저장 잘 되었나 확인하고 데이터 교체하기

In [5]:
#5. 두 파일의 head() 확인

# weather.csv 읽기
weather = pd.read_csv('/workspace/Python/final_exam/data/weather.csv', encoding='utf-8')

# measuredData.csv 읽기
measured = pd.read_csv('/workspace/Python/final_exam/data/measuredData.csv', encoding='utf-8')

print("=" * 70)
print("weather.csv 첫 5행")
print("=" * 70)
print(weather.head())

print("\n" + "=" * 70)
print("measuredData.csv 첫 5행")
print("=" * 70)
print(measured.head())

weather.csv 첫 5행
    지점 지점명               일시  기온(°C)  강수량(mm)  풍속(m/s)  습도(%)
0  108  서울  2021-10-01 1:00    18.7      NaN      1.0     85
1  108  서울  2021-10-01 2:00    18.3      NaN      0.3     89
2  108  서울  2021-10-01 3:00    17.7      NaN      2.0     92
3  108  서울  2021-10-01 4:00    17.4      NaN      1.0     91
4  108  서울  2021-10-01 5:00    17.2      NaN      1.3     94

measuredData.csv 첫 5행
              날짜  아황산가스  일산화탄소     오존  이산화질소  PM10  PM2.5
0  2021-10-01 01  0.003    0.6  0.001  0.042  35.0   20.0
1  2021-10-01 02  0.003    0.5  0.003  0.031  31.0   16.0
2  2021-10-01 03  0.003    0.4  0.005  0.023  25.0   13.0
3  2021-10-01 04  0.003    0.4  0.002  0.025  25.0   13.0
4  2021-10-01 05  0.003    0.4  0.002  0.023  27.0   15.0


In [6]:
#6. 두 파일의 shape, info() 확인

print("=" * 70)
print("weather.csv - 정보")
print("=" * 70)
print(f"Shape: {weather.shape}")
print(f"\nData Types:")
print(weather.dtypes)

print("\n" + "=" * 70)
print("measuredData.csv - 정보")
print("=" * 70)
print(f"Shape: {measured.shape}")
print(f"\nData Types:")
print(measured.dtypes)


weather.csv - 정보
Shape: (744, 7)

Data Types:
지점           int64
지점명         object
일시          object
기온(°C)     float64
강수량(mm)    float64
풍속(m/s)    float64
습도(%)        int64
dtype: object

measuredData.csv - 정보
Shape: (744, 7)

Data Types:
날짜        object
아황산가스    float64
일산화탄소    float64
오존       float64
이산화질소    float64
PM10     float64
PM2.5    float64
dtype: object


In [7]:
#7. 두 파일 비교 분석
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print("=" * 70)
print("두 파일 비교 분석")
print("=" * 70)

print("\n[공통점]")
print(f"- 둘 다 shape: {weather.shape}")
print(f"- 둘 다 744행 (같은 기간의 시계열 데이터)")

print("\n[weather.csv 컬럼]")
for i, col in enumerate(weather.columns, 1):
    print(f"  {i}. {col}")

print("\n[measuredData.csv 컬럼]")
measured_cols = measured.columns.tolist()
for i, col in enumerate(measured_cols, 1):
    print(f"  {i}. {col}")

print("\n[결측치]")
print("weather.csv:")
for col in weather.columns:
    null_count = weather[col].isnull().sum()
    print(f"    - {col}: {null_count}개")

print("\nmeasuredData.csv:")
for col in measured.columns:
    null_count = measured[col].isnull().sum()
    print(f"    - {col}: {null_count}개")

print("\n[시간 형식 차이]")
print(f"weather.csv 일시: {weather['일시'].iloc[0]} (분까지 있음)")
print(f"measuredData.csv 날짜: {measured['날짜'].iloc[0]} (분 없음)")

두 파일 비교 분석

[공통점]
- 둘 다 shape: (744, 7)
- 둘 다 744행 (같은 기간의 시계열 데이터)

[weather.csv 컬럼]
  1. 지점
  2. 지점명
  3. 일시
  4. 기온(°C)
  5. 강수량(mm)
  6. 풍속(m/s)
  7. 습도(%)

[measuredData.csv 컬럼]
  1. 날짜
  2. 아황산가스
  3. 일산화탄소
  4. 오존
  5. 이산화질소
  6. PM10
  7. PM2.5

[결측치]
weather.csv:
    - 지점: 0개
    - 지점명: 0개
    - 일시: 0개
    - 기온(°C): 0개
    - 강수량(mm): 644개
    - 풍속(m/s): 0개
    - 습도(%): 0개

measuredData.csv:
    - 날짜: 0개
    - 아황산가스: 6개
    - 일산화탄소: 6개
    - 오존: 32개
    - 이산화질소: 6개
    - PM10: 6개
    - PM2.5: 6개

[시간 형식 차이]
weather.csv 일시: 2021-10-01 1:00 (분까지 있음)
measuredData.csv 날짜: 2021-10-01 01 (분 없음)


In [8]:
#8. 두 파일 병합 준비 (weather의 시간 기준으로 통합)

# weather.csv의 '일시' 컬럼을 기준으로 measuredData.csv와 병합
# measuredData.csv의 '날짜' 컬럼은 제거하고 weather의 '일시'를 사용
# why? weather의 '일시'가 분까지 있어 더 세분화 된 데이터 확보가 가능하기 때문

print("=" * 70)
print("두 파일 병합")
print("=" * 70)

# measured에서 '날짜' 컬럼 제거 후 weather와 병합
merged = pd.concat([weather, measured.drop('날짜', axis=1)], axis=1)

print(f"\n병합 완료")
print(f"병합된 데이터 shape: {merged.shape}")
print(f"\n병합된 데이터 컬럼:")
for i, col in enumerate(merged.columns, 1):
    print(f"  {i}. {col}")

print(f"\n병합된 데이터 첫 5행:")
print(merged.head())

두 파일 병합

병합 완료
병합된 데이터 shape: (744, 13)

병합된 데이터 컬럼:
  1. 지점
  2. 지점명
  3. 일시
  4. 기온(°C)
  5. 강수량(mm)
  6. 풍속(m/s)
  7. 습도(%)
  8. 아황산가스
  9. 일산화탄소
  10. 오존
  11. 이산화질소
  12. PM10
  13. PM2.5

병합된 데이터 첫 5행:
    지점 지점명               일시  기온(°C)  강수량(mm)  풍속(m/s)  습도(%)  아황산가스  일산화탄소  \
0  108  서울  2021-10-01 1:00    18.7      NaN      1.0     85  0.003    0.6   
1  108  서울  2021-10-01 2:00    18.3      NaN      0.3     89  0.003    0.5   
2  108  서울  2021-10-01 3:00    17.7      NaN      2.0     92  0.003    0.4   
3  108  서울  2021-10-01 4:00    17.4      NaN      1.0     91  0.003    0.4   
4  108  서울  2021-10-01 5:00    17.2      NaN      1.3     94  0.003    0.4   

      오존  이산화질소  PM10  PM2.5  
0  0.001  0.042  35.0   20.0  
1  0.003  0.031  31.0   16.0  
2  0.005  0.023  25.0   13.0  
3  0.002  0.025  25.0   13.0  
4  0.002  0.023  27.0   15.0  


In [ ]:
# # 9. 병합된 데이터를 새로운 CSV로 저장 (잘 병합되었는지 확인 후 한번만 실행)
# merged.to_csv('merged_data.csv', index=False, encoding='utf-8')
# print(f"\n 저장 완료")